In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn xgboost holidays

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import holidays
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings('ignore')

# Set aesthetic styling for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

In [3]:
data_url = "https://drive.google.com/uc?id=1y61cDyuO9Zrp1fSchWcAmCxk0B6SMx7X"

try:
    print("Loading dataset...")
    import requests
    import zipfile
    import io

    response = requests.get(data_url)
    response.raise_for_status()

    outer_zip_file_object = io.BytesIO(response.content)

    with zipfile.ZipFile(outer_zip_file_object, 'r') as zf_outer:
        nested_zip_files = [name for name in zf_outer.namelist() if name.endswith('.zip')]

        if nested_zip_files:
            print(f"Found nested zip file(s): {nested_zip_files}")
            nested_zip_filename = nested_zip_files[0]

            nested_zip_content = zf_outer.read(nested_zip_filename)
            nested_zip_file_object = io.BytesIO(nested_zip_content)

            with zipfile.ZipFile(nested_zip_file_object, 'r') as zf_inner:
                csv_files_in_inner_zip = [name for name in zf_inner.namelist() if name.endswith('.csv')]
                if not csv_files_in_inner_zip:
                    print(f"No CSV file found in the inner zip archive. Contents: {zf_inner.namelist()}")
                    raise ValueError("No CSV file ending with '.csv' found in the inner zip archive.")

                # Explicitly prioritize loading the training dataset by exact name
                csv_filename_to_load = None
                train_file_exact_name = 'smart-city-traffic-patterns/train_aWnotuB.csv'

                if train_file_exact_name in csv_files_in_inner_zip:
                    csv_filename_to_load = train_file_exact_name
                elif csv_files_in_inner_zip: # Fallback to the first CSV if train file is not found
                    csv_filename_to_load = csv_files_in_inner_zip[0]

                if csv_filename_to_load is None:
                    print(f"No suitable CSV file found in the inner zip archive. Contents: {zf_inner.namelist()}")
                    raise ValueError("No CSV file ending with '.csv' found in the inner zip archive.")

                print(f"Attempting to load CSV file from inner zip: {csv_filename_to_load}")

                with zf_inner.open(csv_filename_to_load) as csv_file:
                    df = pd.read_csv(csv_file, encoding='latin1', sep=',', quotechar='"', engine='python', on_bad_lines='warn')
        else:
            csv_files_in_outer_zip = [name for name in zf_outer.namelist() if name.endswith('.csv')]
            if not csv_files_in_outer_zip:
                print(f"No CSV file found in the zip archive. Contents: {zf_outer.namelist()}")
                raise ValueError("No CSV file ending with '.csv' found in the outer zip archive.")

            # Explicitly prioritize loading the training dataset from outer zip
            csv_filename_to_load = None
            train_file_exact_name_outer = 'smart-city-traffic-patterns/train_aWnotuB.csv' # Assuming similar naming if in outer zip

            if train_file_exact_name_outer in csv_files_in_outer_zip:
                csv_filename_to_load = train_file_exact_name_outer
            elif csv_files_in_outer_zip:
                csv_filename_to_load = csv_files_in_outer_zip[0]

            if csv_filename_to_load is None:
                print(f"No suitable CSV file found in the outer zip archive. Contents: {zf_outer.namelist()}")
                raise ValueError("No CSV file ending with '.csv' found in the outer zip archive.")

            print(f"Attempting to load CSV file from outer zip: {csv_filename_to_load}")

            with zf_outer.open(csv_filename_to_load) as csv_file:
                df = pd.read_csv(csv_file, encoding='latin1', sep=',', quotechar='"', engine='python', on_bad_lines='warn')

    print("Dataset loaded successfully!")
except Exception as e:
    print(f"Error loading dataset from URL: {e}")
    print("Falling back to local test data: /content/test_BdBKkAj.csv")
    df = pd.read_csv("/content/test_BdBKkAj.csv")

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Dataset Info ---")
print(df.info())

df['DateTime'] = pd.to_datetime(df['DateTime'])
df = df.sort_values(by=['Junction', 'DateTime']).reset_index(drop=True)

Loading dataset...
Found nested zip file(s): ['Project9_smart-city-traffic-patterns/Project9_smart-city-traffic-patterns.zip']
Attempting to load CSV file from inner zip: smart-city-traffic-patterns/train_aWnotuB.csv
Dataset loaded successfully!

--- First 5 Rows ---
              DateTime  Junction  Vehicles           ID
0  2015-11-01 00:00:00         1        15  20151101001
1  2015-11-01 01:00:00         1        13  20151101011
2  2015-11-01 02:00:00         1        10  20151101021
3  2015-11-01 03:00:00         1         7  20151101031
4  2015-11-01 04:00:00         1         9  20151101041

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48120 entries, 0 to 48119
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DateTime  48120 non-null  object
 1   Junction  48120 non-null  int64 
 2   Vehicles  48120 non-null  int64 
 3   ID        48120 non-null  int64 
dtypes: int64(3), object(1)
memory usa

In [4]:
print("Columns in df:")
print(df.columns)

Columns in df:
Index(['DateTime', 'Junction', 'Vehicles', 'ID'], dtype='object')


In [5]:
print("\n--- First 5 Rows of /content/test_BdBKkAj.csv ---")
test_df = pd.read_csv('/content/test_BdBKkAj.csv')
print(test_df.head())


--- First 5 Rows of /content/test_BdBKkAj.csv ---


FileNotFoundError: [Errno 2] No such file or directory: '/content/test_BdBKkAj.csv'

In [ ]:

# ==========================================
# 2. FEATURE ENGINEERING (HOLIDAYS & TEMPORAL)
# ==========================================
print("\nExtracting time features and holiday patterns...")

# Extract calendar features
df['Year'] = df['DateTime'].dt.year
df['Month'] = df['DateTime'].dt.month
df['Day'] = df['DateTime'].dt.day
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.dayofweek  # 0 = Monday, 6 = Sunday
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Identify national and public holidays
# Adjust country code as per location (defaulting to standard US/Global calendar)
country_holidays = holidays.US(years=df['Year'].unique())
df['IsHoliday'] = df['DateTime'].dt.date.apply(lambda x: 1 if x in country_holidays else 0)

# Cyclical encoding for Hour and Month (captures time loops effectively for ML models)
df['Hour_Sin'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
df['Hour_Cos'] = np.cos(2 * np.pi * df['Hour'] / 24.0)
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12.0)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12.0)

# Create Lag & Rolling Statistics features per Junction
for lag in [1, 2, 24, 168]:  # 1 hr ago, 2 hrs ago, 1 day ago, 1 week ago
    df[f'Vehicles_Lag_{lag}'] = df.groupby('Junction')['Vehicles'].shift(lag)

for window in [3, 6, 24]:    # Rolling average of past 3, 6, and 24 hours
    df[f'Vehicles_Rolling_Mean_{window}'] = df.groupby('Junction')['Vehicles'].transform(
        lambda x: x.shift(1).rolling(window=window).mean()
    )

# Drop initial NaNs generated by lag/rolling windows
df_model = df.dropna().copy()

print(f"Data shape after feature engineering: {df_model.shape}")

In [ ]:
# ==========================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
print("\nGenerating exploratory visualizations...")

# Plot 1: Overall Traffic Volume across 4 Junctions
plt.figure(figsize=(12, 5))
sns.boxplot(x='Junction', y='Vehicles', data=df_model, palette='Set2')
plt.title("Traffic Density Distribution Across Junctions 1-4")
plt.xlabel("Junction Number")
plt.ylabel("Vehicle Count")
plt.tight_layout()
plt.show()

# Plot 2: Average Hourly Traffic (Workdays vs Weekends & Holidays)
plt.figure(figsize=(14, 6))
sns.lineplot(
    data=df_model, x='Hour', y='Vehicles',
    hue='IsWeekend', style='IsHoliday',
    markers=True, ci=None
)
plt.title("Average Hourly Traffic Pattern: Weekday/Weekend vs Holidays")
plt.xlabel("Hour of the Day (0-23)")
plt.ylabel("Average Vehicle Count")
plt.legend(title="Condition", labels=['Weekday', 'Weekend', 'Non-Holiday', 'Holiday'])
plt.tight_layout()
plt.show()

In [ ]:
print("--- df_model Info ---")
df_model.info()

In [ ]:
# ==========================================
# 4. TRAIN / TEST SPLIT (TIME-SERIES SPLIT)
# ==========================================
# Avoid future data leakage by splitting chronologically
features = [
    'Junction', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek',
    'IsWeekend', 'IsHoliday', 'Hour_Sin', 'Hour_Cos', 'Month_Sin', 'Month_Cos',
    'Vehicles_Lag_1', 'Vehicles_Lag_2', 'Vehicles_Lag_24', 'Vehicles_Lag_168',
    'Vehicles_Rolling_Mean_3', 'Vehicles_Rolling_Mean_6', 'Vehicles_Rolling_Mean_24'
]
target = 'Vehicles'

# Train on 85% earliest data, Test on 15% latest data
split_idx = int(len(df_model) * 0.85)

train_df = df_model.iloc[:split_idx]
test_df = df_model.iloc[split_idx:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"\nTraining set size: {X_train.shape[0]} rows")
print(f"Testing set size:  {X_test.shape[0]} rows")

In [ ]:
# ==========================================
# 5. MODEL TRAINING (XGBOOST REGRESSOR)
# ==========================================
print("\nTraining XGBoost Traffic Forecasting Model...")

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

In [ ]:
# ==========================================
# 6. MODEL EVALUATION
# ==========================================
# Avoid SettingWithCopyWarning by explicitly making a copy
test_df = test_df.copy()
test_df['Predictions'] = model.predict(X_test)

# Calculate metrics overall
rmse = np.sqrt(mean_squared_error(y_test, test_df['Predictions']))
mae = mean_absolute_error(y_test, test_df['Predictions'])
r2 = r2_score(y_test, test_df['Predictions'])

print("\n================ MODEL EVALUATION METRICS ================")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"Mean Absolute Error (MAE):     {mae:.2f}")
print(f"R² Score (Accuracy Metric):    {r2*100:.2f}%")
print("==========================================================")

# Per-Junction Performance Breakdown
print("\n--- Performance Breakdown by Junction ---")
# Use unique junctions present in the test set to avoid indexing missing values
for j in sorted(test_df['Junction'].unique()):
    j_test = test_df[test_df['Junction'] == j]

    if len(j_test) > 0:
        j_rmse = np.sqrt(mean_squared_error(j_test['Vehicles'], j_test['Predictions']))
        j_mae = mean_absolute_error(j_test['Vehicles'], j_test['Predictions'])
        print(f"Junction {j} -> RMSE: {j_rmse:.2f} | MAE: {j_mae:.2f}")

In [ ]:
# ==========================================
# 7. FORECAST VISUALIZATION
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False)
axes = axes.flatten()

for idx, j in enumerate(sorted(df_model['Junction'].unique())):
    j_data = test_df[test_df['Junction'] == j].tail(168)  # Sample last 7 days forecast

    axes[idx].plot(j_data['DateTime'], j_data['Vehicles'], label='Actual Traffic', color='blue', alpha=0.7)
    axes[idx].plot(j_data['DateTime'], j_data['Predictions'], label='Forecasted Traffic', color='orange', linestyle='--')
    axes[idx].set_title(f"Junction {j}: Actual vs Forecasted Traffic (7-Day Sample)")
    axes[idx].set_xlabel("Date")
    axes[idx].set_ylabel("Vehicle Count")
    axes[idx].legend()

plt.tight_layout()
plt.show()



In [ ]:
# ==========================================
# 8. FEATURE IMPORTANCE ANALYSIS
# ==========================================
plt.figure(figsize=(10, 6))
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
importance.plot(kind='barh', color='teal')
plt.title("XGBoost Feature Importance for Smart City Traffic Prediction")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()